In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
CODAPATH = Path("/kaggle/working/codapath")
if (CODAPATH / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(CODAPATH), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "pull", "--ff-only", "origin", REPO_BRANCH])
elif CODAPATH.exists():
    raise RuntimeError(f"{CODAPATH} exists but is not a Git repository")
else:
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(CODAPATH)])
actual_branch = subprocess.check_output(["git", "-C", str(CODAPATH), "branch", "--show-current"], text=True).strip()
assert actual_branch == REPO_BRANCH, (actual_branch, REPO_BRANCH)
print("repo:", CODAPATH, "| branch:", actual_branch)

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

In [ ]:
# ---- EDIT THIS CELL ----
# pathmnist | histoset | skintissue
DATASET = "pathmnist"

# random | coreset | typiclust | activeft | badge | entropy | margin
# codapath | scalpel | nucleus_al | nucleus_coverage | uncertainty_herding | tcm | dropquery | refine
SAMPLER_NAME = "nucleus_coverage"
SEED = 42

# One entry per full budget sweep, run back to back in THIS session so the
# dataset and the DINOv2 feature cache are loaded once instead of once per
# Kaggle session. Use [{}] for a sampler that has no variants.
#
# nucleus_al controlled variants:
# 1) {"cell_source": "crop_dino", "uncertainty_mode": "cell_margin"}
# 2) {"cell_source": "cellvit_embedding", "uncertainty_mode": "disagreement"}
# 3) {"cell_source": "cellvit_embedding", "uncertainty_mode": "fusion_concat"}
# 4) {"cell_source": "cellvit_embedding", "uncertainty_mode": "fusion_add"}
#
# nucleus_coverage controlled variants (the 3 experiments of DESIGN.md 4.3).
# Optional ablation: add {"coverage_source": ..., "missing_impute": "zero"} —
# it writes to a separate `_zero` run name, so it never overwrites these.
SAMPLER_VARIANTS = [
    {"coverage_source": "dino"},
    {"coverage_source": "cellvit"},
    {"coverage_source": "concat"},
]

# Leave RUN_NAME=None to derive a collision-safe name per variant. A fixed
# RUN_NAME is only valid for a single variant (otherwise runs overwrite).
RUN_NAME = None

# DINOv2 cache, already extracted by extract_features.ipynb and attached as a
# Kaggle Dataset. A read-only path is checked below: on a cache miss run.py
# would re-extract and then fail trying to write into /kaggle/input.
FEATURE_DIR = "/kaggle/input/datasets/cryandrrich/nckh2026/features"
# Extraction output must first be saved as a Kaggle Dataset and attached here.
# Example: /kaggle/input/pathmnist-nucleus-cache/nucleus_features
NUCLEUS_FEATURE_DIR = "/kaggle/input/EDIT_NUCLEUS_CACHE_SLUG/nucleus_features"
OUTPUT_DIR = "/kaggle/working/checkpoints"

In [ ]:
import os
from huggingface_hub import snapshot_download

# DINOv2 is public: no placeholder login/token is required.
# Enable Kaggle Internet, or set the model path to a mounted local snapshot.
print("Downloading facebook/dinov2-base...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [ ]:
import yaml
import torch

from run import main

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])
PATHMNIST_PATH  = str(DATA_ROOT / "pathmnist_224.npz")
HISTOSET_PATH   = str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14")
SKINTISSUE_PATH = str(DATA_ROOT / "SkinTissue/SkinTissue/tiles")

DATA_DICT = {
    "pathmnist":  PATHMNIST_PATH,
    "histoset":   HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH,
}

In [ ]:
CONFIG_PATH = "config/config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

assert DATASET in DATA_DICT, DATASET
data_path = Path(DATA_DICT[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert config["cumulative_budget"] == [25, 50, 75, 100, 125, 150, 175, 200]
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running AL"
assert SAMPLER_VARIANTS, "SAMPLER_VARIANTS must hold at least one override dict (use [{}])"
assert RUN_NAME is None or len(SAMPLER_VARIANTS) == 1, (
    "A fixed RUN_NAME with several variants makes every run overwrite the previous one"
)

training_cfg = config.get("training", {})
dataset_info = config["datasets"][DATASET]
base_cfg = dict(config.get("samplers", {}).get(SAMPLER_NAME, {}))
sampler_cfgs = [{**base_cfg, **overrides} for overrides in SAMPLER_VARIANTS]

if SAMPLER_NAME in ("nucleus_al", "nucleus_coverage"):
    nucleus_manifest = (
        Path(NUCLEUS_FEATURE_DIR) / f"{DATASET}_seed{SEED}" / "manifest.json"
    )
    assert nucleus_manifest.is_file(), (
        f"Missing nucleus cache: {nucleus_manifest}. Attach the extraction notebook output "
        "and edit NUCLEUS_FEATURE_DIR in the configuration cell."
    )

# A read-only FEATURE_DIR is only usable when the cache is already there:
# on a miss run.py re-extracts DINOv2 and then fails writing into /kaggle/input.
feature_dir = Path(FEATURE_DIR)
if str(feature_dir).startswith("/kaggle/input"):
    vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
    feature_manifest = (
        feature_dir / f"{DATASET}_seed{SEED}_{vit_name.replace('/', '_')}_manifest.json"
    )
    assert feature_manifest.is_file(), (
        f"Missing DINOv2 feature cache: {feature_manifest}. FEATURE_DIR is read-only, "
        "so attach the extract_features.ipynb output or point FEATURE_DIR at a writable path."
    )
    print("features:", feature_manifest)
else:
    feature_dir.mkdir(parents=True, exist_ok=True)
    print("features: will extract into", feature_dir)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
for cfg in sampler_cfgs:
    print(f"[{SAMPLER_NAME}] sampler_cfg =", cfg)
print("data:", data_path, "| output:", OUTPUT_DIR)

In [ ]:
import time

for index, sampler_cfg in enumerate(sampler_cfgs, start=1):
    started = time.time()
    print("=" * 70)
    print(f"VARIANT {index}/{len(sampler_cfgs)}: {SAMPLER_VARIANTS[index - 1]}")
    print("=" * 70)
    main(
        data_path=str(data_path),
        sampler_name=SAMPLER_NAME,
        num_classes=dataset_info["num_classes"],
        cumulative_budget=config["cumulative_budget"],
        data_descriptions=dataset_info["descriptions"],
        prompt_templates=config["prompt_templates"],
        sampler_cfg=sampler_cfg,
        probe_epochs=training_cfg["probe_epochs"],
        probe_lr=training_cfg["probe_lr"],
        device=torch.device(config["device"]),
        random_seed=SEED,
        save_dir=str(Path(OUTPUT_DIR) / DATASET),
        verbose=True,
        model_cfg=config.get("models", {}),
        feature_cache_dir=FEATURE_DIR,
        nucleus_cache_dir=NUCLEUS_FEATURE_DIR,
        run_name=RUN_NAME,
    )
    print(f"VARIANT {index} finished in {(time.time() - started) / 60:.1f} min")